In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [4]:
# Load IMDB text files manually

def load_imdb_split(folder):
    texts = []
    labels = []

    for label_type in ["pos", "neg"]:
        path = os.path.join(folder, label_type)

        for fname in os.listdir(path):
            if fname.endswith(".txt"):
                with open(os.path.join(path, fname), encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(1 if label_type == "pos" else 0)

    return texts, np.array(labels, dtype=np.float32)


base_dir = "./aclImdb"  # <-- change if needed

train_texts, train_labels = load_imdb_split(os.path.join(base_dir, "train"))
test_texts, test_labels = load_imdb_split(os.path.join(base_dir, "test"))

In [5]:
# Bag-of-words vectorization (10k words)

max_words = 10000

vectorizer = CountVectorizer(
    max_features=max_words,
    binary=True,
    stop_words="english"
)

x_train = vectorizer.fit_transform(train_texts).toarray().astype(np.float32)
x_test = vectorizer.transform(test_texts).toarray().astype(np.float32)

In [6]:
# Validation split

x_train, x_val, y_train, y_val = train_test_split(
    x_train, train_labels, test_size=10000, random_state=42, stratify=train_labels
)

In [7]:
# Convert to tensors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train = torch.tensor(x_train).to(device)
y_train = torch.tensor(y_train).unsqueeze(1).to(device)

x_val = torch.tensor(x_val).to(device)
y_val = torch.tensor(y_val).unsqueeze(1).to(device)

In [8]:
# Criar DataLoaders

from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(x_train, y_train)
val_ds   = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

In [9]:
# Definir o modelo (MLP simples para BoW)

class BoWNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)  # logits
        )
    def forward(self, x):
        return self.net(x)

model = BoWNet(max_words).to(device)

In [10]:
# Loss + optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
# Função de avaliação (accuracy)

def accuracy_from_logits(logits, y_true):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds.eq(y_true)).float().mean().item()

In [12]:
# Loop de treino + validação

epochs = 5
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, epochs+1):
    model.train()
    tr_loss, tr_acc = 0.0, 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        tr_loss += loss.item()
        tr_acc  += accuracy_from_logits(logits.detach(), yb)

    model.eval()
    va_loss, va_acc = 0.0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            va_loss += loss.item()
            va_acc  += accuracy_from_logits(logits, yb)

    tr_loss /= len(train_loader); tr_acc /= len(train_loader)
    va_loss /= len(val_loader);   va_acc /= len(val_loader)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(f"Epoch {epoch}: train_loss={tr_loss:.4f} val_loss={va_loss:.4f} train_acc={tr_acc:.4f} val_acc={va_acc:.4f}")

Epoch 1: train_loss=0.3842 val_loss=0.2963 train_acc=0.8445 val_acc=0.8803
Epoch 2: train_loss=0.1892 val_loss=0.3127 train_acc=0.9311 val_acc=0.8750
Epoch 3: train_loss=0.1267 val_loss=0.3561 train_acc=0.9559 val_acc=0.8743
Epoch 4: train_loss=0.0823 val_loss=0.4093 train_acc=0.9747 val_acc=0.8687
Epoch 5: train_loss=0.0507 val_loss=0.4634 train_acc=0.9866 val_acc=0.8654


In [13]:
# Test set

x_test = torch.tensor(x_test).to(device)
y_test = torch.tensor(test_labels).unsqueeze(1).to(device)

test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=256, shuffle=False)

model.eval()
te_acc = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        te_acc += accuracy_from_logits(model(xb), yb)
te_acc /= len(test_loader)
print("Test accuracy:", te_acc)

Test accuracy: 0.8561463647959183
